In [ ]:
# Accept parameters passed from orchestration notebook via dbutils.notebook.run()
# These simulate DAB variables in the bundle deployment

try:
    # Get parameters from dbutils.widgets (passed by dbutils.notebook.run)
    catalog_name = dbutils.widgets.get("catalog_name")
    schema_prefix = dbutils.widgets.get("schema_prefix")
    print(f"Using parameters from orchestration:")
    print(f"  catalog_name: {catalog_name}")
    print(f"  schema_prefix: {schema_prefix}")
except Exception:
    # Fallback to default values if not called from orchestration
    catalog_name = "dev_catalog"
    schema_prefix = "slv_cdm_hrs"
    print(f"Using default values (not called from orchestration):")
    print(f"  catalog_name: {catalog_name}")
    print(f"  schema_prefix: {schema_prefix}")

This notebook is used to load and test the HRS WAVE table.

**Purpose:** Load the HRS Wave reference table.

**Source Table:** `dev_catalog.brz_raw_hrs.randhrs1992_2022v1`  
**Target Table:** `dev_catalog.slv_cdm_hrs.wave`
**Load Script:** `../../sql/dml/load_hrs_wave_data.sql`
**Validation Script:** `../../sql/validataion/verify_hrs_wave_data.sql`

**Process:**
1. Clear/truncate the HRS WAVE table .
2. Load Wave data rows
3. Validate the table data.
4. Display summary stats.

In [ ]:
# -----------------------------------------------------------------------------
# Initialize Notebook Configuration
# ----------------------------------------------------------------------------
# Variables are received from the first cell (either from orchestration or defaults)

dbutils.widgets.dropdown(
    "truncate_table",
    "true",
    ["true", "false"]
)

TRUNCATE_TABLE = dbutils.widgets.get("truncate_table").lower() == "true"

# Build target table name from parameters
TARGET_TABLE = f"{catalog_name}.{schema_prefix}.dim_wave"

LOAD_SQL = "../../sql/dml/load_hrs_wave_data.sql"

VALIDATION_SQL = "../../sql/validation/validate_hrs_wave_data.sql"

SOURCE_TABLE = "dev_catalog.brz_raw_hrs.randhrs1992_2022v1"

In [ ]:
# Step 1:
# Clear existing wave data if needed (use with caution)
# Uncomment the line below to truncate the table before loading 

if TRUNCATE_TABLE:
    print("======================================================")
    print("Step 1 - TRUNCATE")
    print("======================================================")

    try:
        spark.sql(f"TRUNCATE TABLE {TARGET_TABLE}")
        print("✓ Completed")
    except Exception as e:
        print(f"❌ TRUNCATE failed: {e}")
        raise

else:
    print("Table not found.  Skipping table truncation.")

In [ ]:
# Step 2
# Load distinct data to the TARGET_TABLE

from pathlib import Path
import re

print("======================================================")
print("Step 2 - LOAD DATA")
print("======================================================")
try:
    # Read SQL file
    sql_path = Path(LOAD_SQL)
    sql_text = sql_path.read_text()
    
    # Remove comment lines
    sql_text = "\n".join(line for line in sql_text.splitlines() if not line.strip().startswith("--"))
    
    # Split and execute statements
    statements = [stmt.strip() for stmt in sql_text.split(';') if stmt.strip()]
    
    for i, stmt in enumerate(statements, 1):
        print(f"  Executing statement {i}/{len(statements)}")
        
        # Check if this is an INSERT statement with IDENTIFIER
        if stmt.upper().strip().startswith("INSERT") and "IDENTIFIER" in stmt.upper():
            # Check if INSERT ... VALUES or INSERT ... SELECT
            select_start = stmt.upper().find("SELECT")
            if select_start > 0:
                # INSERT ... SELECT pattern
                select_query = stmt[select_start:]
                df = spark.sql(select_query, args={"catalog_name": catalog_name, "schema_prefix": schema_prefix})
                df.write.mode("append").saveAsTable(TARGET_TABLE)
            else:
                # INSERT ... VALUES pattern - replace IDENTIFIER with actual table name
                stmt_fixed = re.sub(
                    r"IDENTIFIER\s*\(\s*CONCAT\s*\([^)]+\)\s*\)",
                    TARGET_TABLE,
                    stmt,
                    flags=re.IGNORECASE
                )
                spark.sql(stmt_fixed)
        else:
            # For non-INSERT statements (TRUNCATE, etc.), use SQL with parameter binding
            #spark.sql(stmt, args={"catalog_name": catalog_name, "schema_prefix": schema_prefix})
            spark.sql(stmt, args={"catalog_name": 'dev_catalog', "schema_prefix": 'slv_cdm_hrs'})
    
    print("✓ Completed")
except Exception as e:
    print(f"❌ Load failed: {e}")
    raise


In [ ]:
# Step 3: Verify the TARGET_TABLE.
from pathlib import Path

print("======================================================")
print("Step 3 - Validation")
print("======================================================")
try:
    # Read SQL file
    sql_path = Path(VALIDATION_SQL)
    sql_text = sql_path.read_text()
    
    # Split and execute statements with parameter binding
    statements = [stmt.strip() for stmt in sql_text.split(';') if stmt.strip()]
    
    for i, stmt in enumerate(statements, 1):
        print(f"  Executing statement {i}/{len(statements)}")
        result = spark.sql(stmt, args={"catalog_name": catalog_name, "schema_prefix": schema_prefix})
        display(result)
    
    print("✓ Completed")
except Exception as e:
    print(f"❌ Validation failed: {e}")
    raise

In [ ]:
# Step 4 - Display summary statistics

target_count = spark.sql(f"""
    SELECT COUNT(*) as wave_count
    FROM {TARGET_TABLE}
""").collect()[0][0]

print("=" * 60)
print("HRS WAVE DATA LOAD SUMMARY")
print("=" * 60)

print(f"Total records in HRS Wave table:      {target_count}")
print("=" * 60)

if target_count == 16:
    print("✓ SUCCESS: All distinct HRS wave loaded")
else:
    print(f"⚠ WARNING: Mismatch detected. Please review.")